# 07 - CNN on Fashion-MNIST (GPU)

Continues from `05_ann_fashion_mnist_pytorch_gpu.ipynb`. Same dataset, same GPU-aware
setup, same train/eval/save/load structure -- but this time using a **Convolutional
Neural Network** instead of a plain fully-connected ANN.

**Why CNNs for images:** the ANN in notebook 05 flattens every image into a single
784-length vector, throwing away all spatial structure (which pixels are NEXT TO which
others). A CNN's convolutional filters slide across the image and can detect local
patterns (edges, corners, textures) wherever they appear -- usually reaching higher
accuracy with fewer parameters than a comparable ANN on image data.

**What's different from notebook 05:**
1. Images are kept as 2D (28x28) instead of flattened to 784
2. The model uses `nn.Conv2d` + pooling layers before the final fully-connected layers
3. Everything else (Dataset, DataLoader, training loop, save/load, plotting) is the
   same pattern you already know


In [ ]:
# Same imports as notebook 05, plus nothing new -- Conv2d lives in torch.nn already
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Same portable data loading pattern as notebook 05.
CSV_PATH = 'fmnist_small.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded real Fashion-MNIST data from '{CSV_PATH}'")
else:
    print(f"'{CSV_PATH}' not found -- generating a small SYNTHETIC placeholder dataset.")
    rng = np.random.RandomState(42)
    n_samples = 2000
    labels = rng.randint(0, 10, n_samples)
    class_patterns = rng.randint(50, 200, size=(10, 784))
    pixels = class_patterns[labels] + rng.randint(-30, 30, size=(n_samples, 784))
    pixels = np.clip(pixels, 0, 255)
    df = pd.DataFrame(pixels, columns=[f'pixel{i}' for i in range(784)])
    df.insert(0, 'label', labels)

df.fillna(0, inplace=True)
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test = X_train/255.0, X_test/255.0
print("train:", X_train.shape, " test:", X_test.shape)

## The Key Difference: Keep the 2D Image Shape

The ANN's `CustomDataset` (notebook 05) returned each image as a flat 784-length
vector. A CNN's `Conv2d` layer expects a shape of `(channels, height, width)` --
for grayscale Fashion-MNIST that's `(1, 28, 28)`. We reshape each row back into that
shape inside the Dataset instead of leaving it flat.


In [ ]:
class CNNDataset(Dataset):
  def __init__(self, features, labels):
    # Reshape each flat 784-length row back into (1, 28, 28):
    #   1  = number of channels (grayscale, so just 1 -- RGB images would use 3)
    #   28 = height, 28 = width
    self.features = torch.tensor(features, dtype=torch.float32).reshape(-1, 1, 28, 28)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

train_dataset = CNNDataset(X_train, y_train)
test_dataset = CNNDataset(X_test, y_test)

print("one training image's shape:", train_dataset[0][0].shape, "(channels, height, width)")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)
print("batches per epoch:", len(train_loader))

## The CNN Architecture

A small, standard CNN pattern: two convolution+pool blocks (extracting spatial
features), then flatten and feed into fully-connected layers (same as the ANN's
final layers) to produce the 10 class scores.

```
input (1,28,28)
  -> Conv2d(1->16, 3x3) -> ReLU -> MaxPool2d(2)   -> (16,14,14)
  -> Conv2d(16->32,3x3) -> ReLU -> MaxPool2d(2)   -> (32, 7, 7)
  -> Flatten                                       -> (32*7*7,)
  -> Linear(32*7*7 -> 64) -> ReLU
  -> Linear(64 -> 10)                              -> class scores
```


In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self, num_classes=10):
    super().__init__()
    self.conv_block = nn.Sequential(
        # First conv block: 1 input channel (grayscale) -> 16 feature maps
        nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),  # keeps 28x28
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),   # 28x28 -> 14x14 (halves height & width)

        # Second conv block: 16 -> 32 feature maps
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1), # keeps 14x14
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),   # 14x14 -> 7x7
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),                   # (32, 7, 7) -> (32*7*7,) = (1568,)
        nn.Linear(32 * 7 * 7, 64),
        nn.ReLU(),
        nn.Linear(64, num_classes),
    )

  def forward(self, x):
    x = self.conv_block(x)
    x = self.classifier(x)
    return x

# Sanity-check the shapes flow correctly before training
sample_batch, _ = next(iter(train_loader))
test_model = SimpleCNN()
test_output = test_model(sample_batch)
print("input batch shape: ", sample_batch.shape)
print("output batch shape:", test_output.shape, "(batch_size, num_classes)")

## Training -- Same Pattern as Notebook 05

Identical training loop structure to the ANN notebook: `model.train()`, forward pass,
loss, `zero_grad()`, `backward()`, `step()`, then `model.eval()` for the test-loss
pass. The ONLY thing that changed is which `model` class we're training.


In [ ]:
learning_rate = 0.01
epochs = 15   # CNNs typically need fewer epochs than a plain ANN for the same task

model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)   # Adam tends to work well for CNNs

train_losses, test_losses = [], []

for epoch in range(epochs):
  model.train()
  total_train_loss = 0
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)
    loss = criterion(outputs, batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_train_loss += loss.item()

  avg_train_loss = total_train_loss / len(train_loader)
  train_losses.append(avg_train_loss)

  model.eval()
  total_test_loss = 0
  with torch.no_grad():
    for batch_features, batch_labels in test_loader:
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
      outputs = model(batch_features)
      total_test_loss += criterion(outputs, batch_labels).item()

  avg_test_loss = total_test_loss / len(test_loader)
  test_losses.append(avg_test_loss)

  print(f'Epoch: {epoch + 1:2d} , Train Loss: {avg_train_loss:.4f} , Test Loss: {avg_test_loss:.4f}')

## Evaluate: Test Accuracy


In [ ]:
model.eval()
total, correct = 0, 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

cnn_test_accuracy = correct / total
print(f"CNN test accuracy: {cnn_test_accuracy:.4f}")

## Save & Load -- Same Pattern as Notebook 05


In [ ]:
MODEL_PATH = 'fashion_mnist_cnn_model.pth'
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")
print(f"File size: {os.path.getsize(MODEL_PATH) / 1024:.2f} KB")

# Reload and verify it produces the same accuracy
loaded_model = SimpleCNN(num_classes=10)
loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()

total, correct = 0, 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    outputs = loaded_model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f"Test accuracy (loaded model): {correct/total:.4f}  (should match the number above)")

## CNN vs. ANN: Parameter Count Comparison

A common surprise: a CNN can reach comparable or better accuracy with **far fewer
parameters** than a fully-connected ANN on the same image task, because convolution
filters are *reused* across every position in the image (parameter sharing) instead
of needing a unique weight for every pixel-to-neuron connection.


In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

# The ANN architecture from notebook 05, for comparison
class ReferenceANN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )
  def forward(self, x):
    return self.model(x)

ann_reference = ReferenceANN(784)
cnn_params = count_params(model)
ann_params = count_params(ann_reference)

print(f"CNN (this notebook) parameters: {cnn_params:,}")
print(f"ANN (notebook 05)   parameters: {ann_params:,}")
print(f"CNN uses {ann_params/cnn_params:.2f}x {'fewer' if cnn_params < ann_params else 'more'} parameters")